In [0]:
# Create a text widget named "catalog" with default value "new_catalog"
dbutils.widgets.text("catalog", "new_catalog")

# Retrieve the value entered in the "catalog" widget, strip whitespace, and store in variable Catalog1
Catalog1 = dbutils.widgets.get("catalog").strip()

# Create a text widget named "schema" with default value "default_schema"
dbutils.widgets.text("schema", "default_schema")

# Retrieve the value entered in the "schema" widget, strip whitespace, and store in variable Schema1
Schema1 = dbutils.widgets.get("schema").strip()

In [0]:
import json

# Run the common configuration notebook with a 360-second timeout
# Pass in dynamic parameters for catalog and schema (from widgets)
json_obj = dbutils.notebook.run(
    "/Workspace/Users/viggneshwar@gmail.com/databricks/Logistics/Project/Generic_Function/common_config_nb",
    360,
    {"catalog_new": Catalog1, "schema_new": Schema1}
)

# Parse the JSON string returned by the notebook into a Python dictionary
config_dict = json.loads(json_obj)

# Extract key configuration values from the dictionary
bronze_path = config_dict["bronze_path"]        # Path for bronze layer data
source_path = config_dict["sourcedata_path"]    # Path for source/raw data
gold_path   = config_dict["gold_path"]          # Path for gold layer data
gold_db     = config_dict["gold_db"]            # Database/schema for gold layer tables
silver_db   = config_dict["silver_db"]          # Database/schema for silver layer tables

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_db}.daily_dispatch_schedule AS
SELECT
    *
FROM {gold_db}.logistics_shipment_gold_curated_tbl
ORDER BY shipment_date ASC, shipment_cost DESC
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_db}.critical_delays AS
SELECT *
FROM {gold_db}.logistics_shipment_gold_curated_tbl
WHERE shipment_status = 'DELAYED'
ORDER BY shipment_date ASC, shipment_cost DESC
LIMIT 10
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_db}.staff_with_shipments AS
SELECT
    *
FROM {gold_db}.staff_gold_tbl t1
INNER JOIN {gold_db}.logistics_shipment_gold_curated_tbl t2
    ON t1.shipment_id = t2.log_shipment_id
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_db}.invalid_driver_to_shipments AS
SELECT 
    t2.* 
FROM {gold_db}.staff_gold_tbl t1
RIGHT JOIN {gold_db}.logistics_shipment_gold_curated_tbl t2
    ON t1.shipment_id = t2.log_shipment_id
WHERE t1.shipment_id IS NULL
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_db}.geo_tagged_staff_data AS
SELECT 
    t1.*,            -- All staff attributes from the gold staff table
    t2.latitude,     -- Latitude from the master city reference
    t2.longitude     -- Longitude from the master city reference
FROM {gold_db}.staff_gold_tbl t1
LEFT JOIN {silver_db}.master_city_df_tbl t2
    ON t1.origin_hub_city = t2.city_name
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_db}.multi_level_report AS
SELECT 
    SUM(shipment_cost) AS total_cost,   -- Aggregated shipment cost
    origin_hub_city,                    -- Dimension 1: hub city
    vehicle_type                        -- Dimension 2: vehicle type
FROM {gold_db}.staff_with_shipments
GROUP BY CUBE(origin_hub_city, vehicle_type)
""")